# Notebook — Tracking Pipeline (no stitching)

Identical to `01_tracking_pipeline.ipynb` but with no Hungarian stitching.
`stitched_id` equals `orig_id` — OC-SORT track IDs are used directly.

New tracker features active:
- OCM direction term (velocity consistency in association)
- Vial-aware hard constraint (no cross-vial matches)
- Behavioral consistency bonus (speed + scale plausibility)

## Stages
1. Setup & configuration
2. (Optional) Background subtraction
3. Draw vial ROIs
4. RF-DETR + OC-SORT tracking → wide CSV
5. Passthrough (no stitching) → stitched long CSV with stitched_id == orig_id
6. Vial assignment + compact IDs → compact_tracks.csv
7. Overlay video rendering

**Replace all `PLACEHOLDER` paths with your actual file paths.**

In [1]:
import sys
sys.path.insert(0, '..')

import json
import os
import re
import cv2
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Video

from src.preprocessing import preprocess_bgsub_gui
from src.metrics import run_diagnostics, compute_stitching_objectives, print_stitching_objectives
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long
from src.roi import draw_and_save_vial_rois, assign_vials_and_compact_ids
from src.visualization import render_vial_overlay_video, render_raw_overlay_video, render_detections_video
from utils import save_run_params

## 1 â€” Configuration

Set your paths and Roboflow credentials here.

In [ ]:
# ---- EDIT THESE ----
RAW_VIDEO = r"C:\Users\emmav\Downloads\superfly\2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m\41 DPE\004\2024-03-11_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_41d_004-converted.mp4"
MODEL_ID  = "flies-123/1"   # e.g. "flies-123/1"

# Load API key from creds_config.yaml (not committed to git)
with open("../creds_config.yaml", "r") as f:
    creds_config = yaml.safe_load(f)
API_KEY = creds_config["API_KEY"]

# Load defaults from config.yaml (override below if needed)
with open("../config.yaml") as _f:
    _cfg = yaml.safe_load(_f)
_t = _cfg.get("tracker", {})
_s = _cfg.get("stitching", {})
_p = _cfg.get("preprocessing", {})

detection_confidence_rfdetr = _t.get("detection_confidence_rfdetr", 0.4)
confidence              = _t.get("confidence", 0.1)
lost_track_buffer       = _t.get("lost_track_buffer", 90)
min_matching_threshold  = _t.get("minimum_matching_threshold", 0.2)
min_consecutive_frames  = _t.get("minimum_consecutive_frames", 3)
asso_func               = _t.get("asso_func", "diou")
brownian_pos_noise      = _t.get("brownian_pos_noise", 1.0)
aspect_weight           = _t.get("aspect_weight", 0.05)
behavioral_weight       = _t.get("behavioral_weight", 0.05)
jump_factor             = _t.get("jump_factor", 2.0)
jump_iou_threshold      = _t.get("jump_iou_threshold", 0.05)
jump_inertia            = _t.get("jump_inertia", 0.05)
bg_gain                 = _p.get("bg_gain", 1.2)
bg_white_level          = _p.get("bg_white_level", 245)
bg_percentile           = _p.get("bg_percentile", 85.0)
bg_sample_stride        = _p.get("bg_sample_stride", 1)
default_end             = _p.get("default_end", 700)

# Extract short label from the "N DPE/NNN" directory convention in the video path.
_m = re.search(r'(\d+)\s+DPE[/\\](\d+)', RAW_VIDEO)
short_name = f"{_m.group(1)}DPE_n{_m.group(2).zfill(3)}" if _m else Path(RAW_VIDEO).stem[:20]

# Auto-increment output directory
_outputs_root = Path("../outputs")
_outputs_root.mkdir(parents=True, exist_ok=True)
_existing = [d for d in _outputs_root.iterdir() if d.is_dir() and d.name.startswith("run_")]
_next_n = max((int(d.name.split("_")[1]) for d in _existing if d.name.split("_")[1].isdigit()), default=0) + 1
_dir_name = f"run_{_next_n}_{_m.group(1)}DPE_n{_m.group(2).zfill(3)}" if _m else f"run_{_next_n}"
OUTPUT_PATH = str(_outputs_root / _dir_name)

os.makedirs(OUTPUT_PATH, exist_ok=True)

import shutil
_dest_video = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).name)
if not os.path.exists(_dest_video):
    try:
        os.link(RAW_VIDEO, _dest_video)
    except OSError:
        shutil.copy2(RAW_VIDEO, _dest_video)
PATH_TO_VID = RAW_VIDEO

print("Output dir:", OUTPUT_PATH)
print("Short name:", short_name)
print(f"detection_confidence_rfdetr={detection_confidence_rfdetr}, asso_func={asso_func}")
print(f"aspect_weight={aspect_weight}, behavioral_weight={behavioral_weight}")
print(f"jump_factor={jump_factor}, jump_iou_threshold={jump_iou_threshold}, jump_inertia={jump_inertia}")
_cap = cv2.VideoCapture(RAW_VIDEO)
save_run_params(OUTPUT_PATH, "config", {
    "video": RAW_VIDEO, "output_dir": OUTPUT_PATH, "short_name": short_name,
    "video_fps": _cap.get(cv2.CAP_PROP_FPS),
    "video_width": int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "video_height": int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "video_frames": int(_cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    "tracker": {"detection_confidence_rfdetr": detection_confidence_rfdetr,
                 "confidence": confidence, "lost_track_buffer": lost_track_buffer,
                 "min_matching_threshold": min_matching_threshold,
                 "min_consecutive_frames": min_consecutive_frames, "asso_func": asso_func,
                 "brownian_pos_noise": brownian_pos_noise,
                 "aspect_weight": aspect_weight, "behavioral_weight": behavioral_weight,
                 "jump_factor": jump_factor, "jump_iou_threshold": jump_iou_threshold,
                 "jump_inertia": jump_inertia},
    "preprocessing": {"bg_gain": bg_gain, "bg_white_level": bg_white_level,
                       "bg_percentile": bg_percentile, "bg_sample_stride": bg_sample_stride},
})
_cap.release()

Output dir: ..\outputs\run_88_28DPE_n003
Short name: 28DPE_n003
detection_confidence_rfdetr=0.4, asso_func=diou
aspect_weight=0.05, behavioral_weight=0.05
jump_factor=2.0, jump_iou_threshold=0.05, jump_inertia=0.05


## 2 â€” (Optional) Background subtraction

Opens a GUI: draw a crop ROI and choose a frame range.
The output is a `_pp.mp4` file with the **85th-percentile** background subtracted.

In [3]:
ROI_LIBRARY = Path("../roi_library.json")
_video_key = Path(RAW_VIDEO).stem

# Load existing library (or start fresh)
if ROI_LIBRARY.exists():
    with open(ROI_LIBRARY) as f:
        _library = json.load(f)
else:
    _library = {}

_crop_params = None
_use_saved_roi = _cfg.get("roi", {}).get("use_saved_roi", True)
preprocess = True  # set to False to skip bg subtraction entirely

if preprocess:
    pp_out = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).stem + "_pp.mp4")
    _stored_crop = _library.get(_video_key, {}).get("preprocessing") if _use_saved_roi else None

    if _use_saved_roi and _stored_crop is not None:
        print(f"Found stored preprocessing params for: {_video_key}")
    else:
        if not _use_saved_roi:
            print("use_saved_roi=False - opening preprocessing GUI...")
        else:
            print(f"No stored preprocessing params for: {_video_key} - opening GUI...")

    pp_path, _crop_params = preprocess_bgsub_gui(
        video_path=RAW_VIDEO,
        out_mp4=pp_out,
        gain=bg_gain,
        white_level=bg_white_level,
        bg_sample_stride=bg_sample_stride,
        bg_percentile=bg_percentile,
        crop_params=_stored_crop if _use_saved_roi else None,
    )
    PATH_TO_VID = Path(pp_path)

    # Save crop params + full video path to library
    if _video_key not in _library:
        _library[_video_key] = {}
    _library[_video_key]["preprocessing"] = _crop_params
    _library[_video_key]["video_path"] = RAW_VIDEO
    ROI_LIBRARY.parent.mkdir(parents=True, exist_ok=True)
    with open(ROI_LIBRARY, "w") as f:
        json.dump(_library, f, indent=2)
    print("Preprocessing params saved to library.")

    # Save crop_roi.json to this run folder (allows skipping GUI on re-runs)
    with open(os.path.join(OUTPUT_PATH, "crop_roi.json"), "w") as _f:
        json.dump(_crop_params, _f, indent=2)

save_run_params(OUTPUT_PATH, "preprocessing",
                {"video_pp": str(PATH_TO_VID), "crop_params": _crop_params})


Found stored preprocessing params for: 2024-02-27_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_28d_003-converted
Using stored crop params: x=149, y=172, w=703, h=402, frames=0–393
Saved bgsub video: ..\outputs\run_88_28DPE_n003\2024-02-27_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_28d_003-converted_pp.mp4
Background (85.0th percentile) from 365 frames (stride=1).
Preprocessing params saved to library.


## 3 â€” Draw vial ROIs

Opens an OpenCV GUI on frame 0: drag rectangles around each vial.
Press **q** when all 6 ROIs are drawn. Saved to `vial_rois.json`.

This is a one-time step â€” reuse the JSON for the same experimental setup.

In [4]:
ROI_JSON = os.path.join(OUTPUT_PATH, "vial_rois.json")
_use_saved_roi = _cfg.get("roi", {}).get("use_saved_roi", True)
_stored_vials = _library.get(_video_key, {}).get("vial_rois")

if _use_saved_roi and _stored_vials is not None:
    print(f"Found stored vial ROIs for: {_video_key}")
    _vials = {k: tuple(v) for k, v in _stored_vials.items()}
    with open(ROI_JSON, "w") as f:
        json.dump({k: list(v) for k, v in _vials.items()}, f, indent=2)
    print(f"Loaded {len(_vials)} vials from library.")
else:
    if not _use_saved_roi:
        print("use_saved_roi=False â€” opening GUI...")
    else:
        print(f"No stored vial ROIs for: {_video_key} â€” opening GUI...")
    _vials = draw_and_save_vial_rois(video_path=str(PATH_TO_VID), roi_json_path=ROI_JSON)

    # Save to library
    if _video_key not in _library:
        _library[_video_key] = {}
    _library[_video_key]["vial_rois"] = {k: list(v) for k, v in _vials.items()}
    ROI_LIBRARY.parent.mkdir(parents=True, exist_ok=True)
    with open(ROI_LIBRARY, "w") as f:
        json.dump(_library, f, indent=2)
    print("Vial ROIs saved to library.")

save_run_params(OUTPUT_PATH, "roi", {k: list(v) for k, v in _vials.items()})

Found stored vial ROIs for: 2024-02-27_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_28d_003-converted
Loaded 6 vials from library.


## 4 â€” RF-DETR + OC-SORT tracking

Runs the detector + tracker on every frame and writes a wide CSV.
This is the most time-consuming step. 

In [ ]:
WIDE_CSV = os.path.join(OUTPUT_PATH, "tracks_wide_format.csv")
DET_LOG_CSV = os.path.join(OUTPUT_PATH, "detections_raw.csv")

# ── Detection cache ───────────────────────────────────────────────────────────
# Set CACHED_DETS to a previous run's detections_raw.csv to skip RF-DETR.
# Leave as None to run inference and save fresh detections to DET_LOG_CSV.
CACHED_DETS = r"C:\Users\emmav\Downloads\superfly\outputs\run_68_41DPE_n004\detections_raw.csv"

_det_source = CACHED_DETS if (CACHED_DETS and os.path.exists(CACHED_DETS)) else DET_LOG_CSV
if CACHED_DETS and os.path.exists(CACHED_DETS):
    print(f"Using cached detections: {CACHED_DETS}")
else:
    print("No cache found — running RF-DETR inference")

df_wide, tracker = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=WIDE_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    detection_confidence_rfdetr=detection_confidence_rfdetr,
    confidence=confidence,
    lost_track_buffer=lost_track_buffer,
    minimum_matching_threshold=min_matching_threshold,
    minimum_consecutive_frames=min_consecutive_frames,
    asso_func=asso_func,
    brownian_pos_noise=brownian_pos_noise,
    det_log_csv=_det_source,
    vial_rois=_vials,
    aspect_weight=aspect_weight,
    behavioral_weight=behavioral_weight,
    jump_factor=jump_factor,
    jump_iou_threshold=jump_iou_threshold,
    jump_inertia=jump_inertia,
    max_frames=None,
)

print(df_wide.shape)
save_run_params(OUTPUT_PATH, "tracker_output", {
    "wide_csv": WIDE_CSV, "frames": int(df_wide.shape[0]), "track_count": int(df_wide.shape[1] - 1),
})
df_wide.head()

with open(os.path.join(OUTPUT_PATH, "tracker_log.json"), "w") as _f:
    json.dump({
        "detection_log":     tracker.detection_log,
        "suppressed_tracks": tracker.suppressed_tracks,
        "min_hits":          tracker.min_hits,
        "max_age":           tracker.max_age,
    }, _f)

render_detections_video(
    video_path=str(PATH_TO_VID),
    det_log_csv=_det_source,
    out_mp4=os.path.join(OUTPUT_PATH, f"{short_name}_detections_RF-DETR.mp4"),
)

No cache found — running RF-DETR inference


TypeError: remove: path should be string, bytes or os.PathLike, not NoneType

Exception ignored in: 'scipy._lib.messagestream.MessageStream.__dealloc__'
Traceback (most recent call last):
  File "scipy/_lib/messagestream.pyx", line 91, in scipy._lib.messagestream.MessageStream.close
TypeError: remove: path should be string, bytes or os.PathLike, not NoneType


KeyboardInterrupt: 

In [ ]:
# Quick mid-pipeline check: are detections reaching the tracker?
# No compact IDs yet (stitching hasn't run), so no output report saved here.
run_diagnostics(
    tracker     = tracker,
    df_wide     = df_wide,
    df_stitched = None,
    n_expected  = 42,
    fps         = 30,
    config      = _cfg,
)

  RUN CONFIGURATION
{
  "tracker": {
    "detection_confidence_rfdetr": 0.4,
    "confidence": 0.55,
    "track_activation_threshold": 0.1,
    "lost_track_buffer": 180,
    "minimum_matching_threshold": 0.1,
    "minimum_consecutive_frames": 3,
    "min_area": 20,
    "asso_func": "diou",
    "brownian_pos_noise": 15,
    "aspect_weight": 0.05,
    "behavioral_weight": 0.05,
    "jump_factor": 2.0,
    "jump_iou_threshold": 0.05,
    "jump_inertia": 0.05
  },
  "stitching": {
    "stitching_mode": "per_vial",
    "stop_mode": "converge",
    "max_rounds": 10,
    "w_under": 15,
    "w_over": 2.0,
    "vial_count_cap": 7,
    "general_count_cap": 56,
    "fps": 30,
    "pause_threshold": 1.0,
    "edge_fraction": 0.1,
    "min_points_for_scale": 10,
    "expected_per_vial": 7,
    "short_track_frac": 0.1,
    "link_score_weights": {
      "extrap": 0.4,
      "direction": 0.3,
      "behavioral": 0.3
    },
    "direction_weights": {
      "heading_vs_gap": 0.7,
      "overall_vs_overa

## 5 — No stitching (passthrough)

OC-SORT track IDs are used directly. `stitched_id` is set equal to `orig_id`.
`wide_to_long` converts the wide CSV to long format as usual.

In [ ]:
STITCHED_CSV = os.path.join(OUTPUT_PATH, "tracks_xy_stitched_long.csv")
LONG_CSV     = os.path.join(OUTPUT_PATH, "tracks_long_format.csv")

with open(ROI_JSON) as f:
    vial_rois = {k: tuple(v) for k, v in json.load(f).items()}

long_df = wide_to_long(pd.read_csv(WIDE_CSV), out_csv=LONG_CSV)

# No stitching: stitched_id == orig_id
stitched_df = long_df.copy()
stitched_df["stitched_id"] = stitched_df["orig_id"]
stitched_df.to_csv(STITCHED_CSV, index=False)

print(f"Track IDs (no stitching): {stitched_df['orig_id'].nunique()}")
save_run_params(OUTPUT_PATH, "stitching_output", {
    "stitched_csv": STITCHED_CSV,
    "stitched_ids": int(stitched_df["stitched_id"].nunique()),
    "original_ids": int(stitched_df["orig_id"].nunique()),
})

Track IDs (no stitching): 44


## 6 â€” Vial assignment + compact IDs

Assigns each point to a vial using the ROI JSON, then assigns compact sequential IDs
(left â†’ right within each vial).

In [ ]:
COMPACT_CSV = os.path.join(OUTPUT_PATH, "compact_tracks.csv")

df_compact = assign_vials_and_compact_ids(
    stitched_csv=STITCHED_CSV,
    roi_json=ROI_JSON,
    out_csv=COMPACT_CSV,
    fps=_s.get("fps", 30),
)

print(df_compact.shape)
save_run_params(OUTPUT_PATH, "compact", {"csv": COMPACT_CSV, "rows": int(df_compact.shape[0])})
df_compact.head()

(10769, 8)


,frame,orig_id,x,y,stitched_id,vial_id,compact_id,fps
0,0,id1,535.26,238.35,id1,vial5,40,30.0
1,1,id1,536.26,237.39,id1,vial5,40,30.0
2,2,id1,537.00,236.07,id1,vial5,40,30.0
3,3,id1,538.23,234.91,id1,vial5,40,30.0
4,4,id1,539.08,234.09,id1,vial5,40,30.0


In [ ]:
df_wide = pd.read_csv(WIDE_CSV)
num_frames = int(df_wide["frame"].max()) + 1

stitching_objectives = compute_stitching_objectives(
    df_stitched       = stitched_df,
    vial_rois         = vial_rois,
    num_frames        = num_frames,
    expected_per_vial = _s.get("expected_per_vial", 7),
    short_frac        = _s.get("short_track_frac", 0.10),
)
print_stitching_objectives(stitching_objectives)
save_run_params(OUTPUT_PATH, "stitching_objectives", {k: float(v) for k, v in stitching_objectives.items()})

run_diagnostics(
    tracker              = tracker,
    df_wide              = df_wide,
    df_stitched          = stitched_df,
    df_compact           = df_compact,
    n_expected           = _s.get("expected_per_vial", 7) * len(vial_rois),
    fps                  = _s.get("fps", 30),
    vial_rois            = vial_rois,
    config               = _cfg,
    output_dir           = OUTPUT_PATH,
    stitching_objectives = stitching_objectives,
)

  STITCHING QUALITY OBJECTIVES
  vial_count_error      : 7.0  (0 = perfect)
  per_id_coverage_loss  : 120.2  frames/fly
  short_track_count     : 5
  per_frame_id_variance : 6.577
  RUN CONFIGURATION
{
  "tracker": {
    "detection_confidence_rfdetr": 0.4,
    "confidence": 0.55,
    "track_activation_threshold": 0.1,
    "lost_track_buffer": 180,
    "minimum_matching_threshold": 0.1,
    "minimum_consecutive_frames": 3,
    "min_area": 20,
    "asso_func": "diou",
    "brownian_pos_noise": 15,
    "aspect_weight": 0.05,
    "behavioral_weight": 0.05,
    "jump_factor": 2.0,
    "jump_iou_threshold": 0.05,
    "jump_inertia": 0.05
  },
  "stitching": {
    "stitching_mode": "per_vial",
    "stop_mode": "converge",
    "max_rounds": 10,
    "w_under": 15,
    "w_over": 2.0,
    "vial_count_cap": 7,
    "general_count_cap": 56,
    "fps": 30,
    "pause_threshold": 1.0,
    "edge_fraction": 0.1,
    "min_points_for_scale": 10,
    "expected_per_vial": 7,
    "short_track_frac": 0.1,
   

INFO | Chromium init'ed with kwargs {}
INFO | Found chromium path: C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe
INFO | Temp directory created: C:\Users\emmav\AppData\Local\Temp\tmpzvbkzvm9.
INFO | Opening browser.
INFO | Temp directory created: C:\Users\emmav\AppData\Local\Temp\tmpgzk85at_.
INFO | Temporary directory at: C:\Users\emmav\AppData\Local\Temp\tmpgzk85at_
INFO | Conforming 1 to file:///C:/Users/emmav/AppData/Local/Temp/tmpzvbkzvm9/index.html
INFO | Waiting on all navigates
INFO | All navigates done, putting them all in queue.
INFO | Getting tab from queue (has 1)
INFO | Got 2C96
INFO | Processing XY_trajectories_raw_tracker_IDs_left_vs_compact_IDs_after_stitching_right.png
INFO | Sending big command for XY_trajectories_raw_tracker_IDs_left_vs_compact_IDs_after_stitching_right.png.
INFO | Sent big command for XY_trajectories_raw_tracker_IDs_left_vs_compact_IDs_after_stitching_right.png.
INFO | Reloading tab 2C96 before return.
INFO | Putting tab 2C96 back (que

Report saved: ..\outputs\run_77_28DPE_n003\metrics_report.html
           +  ..\outputs\run_77_28DPE_n003\metrics_report.md


## 7 â€” Overlay video

Renders each fly as a coloured dot on the original video.

In [ ]:
RAW_OVERLAY_MP4 = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_raw_ocsort.mp4")
OVERLAY_MP4     = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_vials_shaded.mp4")

# Pick the overlay substrate from config.yaml:visualization.overlay_source.
# Kept separate from PATH_TO_VID, which is the tracker input (_pp after Stage 2).
# In raw_cropped mode, _resolve_overlay_source in src/visualization.py looks up
# crop_params in roi_library.json by Path(video).stem â€” so we must pass RAW_VIDEO
# (whose stem matches the library key), not the _pp path.
_overlay_mode = _cfg.get("visualization", {}).get("overlay_source", "raw_cropped").lower()
OVERLAY_VIDEO = RAW_VIDEO if _overlay_mode == "raw_cropped" else str(PATH_TO_VID)
print(f"overlay_source={_overlay_mode}  â†’  substrate: {OVERLAY_VIDEO}")

render_raw_overlay_video(
    video_path=OVERLAY_VIDEO,
    csv_path=LONG_CSV,
    out_mp4=RAW_OVERLAY_MP4,
)

render_vial_overlay_video(
    video_path=OVERLAY_VIDEO,
    csv_path=COMPACT_CSV,
    out_mp4=OVERLAY_MP4,
)

save_run_params(OUTPUT_PATH, "outputs", {"raw_overlay": RAW_OVERLAY_MP4, "overlay": OVERLAY_MP4})
Video(RAW_OVERLAY_MP4, width=800)

overlay_source=raw_cropped  â†’  substrate: C:\Users\emmav\Downloads\superfly\2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m\28 DPE\003\2024-02-27_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_28d_003-converted.mp4
Saved raw overlay video: ..\outputs\run_77_28DPE_n003\28DPE_n003_overlay_raw_ocsort.mp4
Saved overlay video: ..\outputs\run_77_28DPE_n003\28DPE_n003_overlay_vials_shaded.mp4


## 8 — Jump round ablation

Runs the tracker **twice** on the same detection cache (run_68, 41DPE_n004),
changing only whether the jump round is active. All other params match run_68 exactly:
`confidence=0.55, brownian=15, asso_func=diou, aspect_weight=0, behavioral_weight=0`.

This isolates the effect of the inflated second-round search on raw track count and coverage.

In [ ]:
import tempfile, os, json
import pandas as pd
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long

_DET_CACHE  = "../outputs/run_68_41DPE_n004/detections_raw.csv"
_PP_VIDEO   = "../outputs/run_68_41DPE_n004/2024-03-11_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_41d_004-converted_pp.mp4"
_ROI_JSON   = "../outputs/run_68_41DPE_n004/vial_rois.json"
_N_EXPECTED = 42  # 6 vials x 7 flies

with open(_ROI_JSON) as f:
    _ablation_vials = {k: tuple(v) for k, v in json.load(f).items()}

_base_kwargs = dict(
    video_path                  = _PP_VIDEO,
    api_key                     = API_KEY,
    model_id                    = MODEL_ID,
    detection_confidence_rfdetr = 0.4,
    confidence                  = 0.55,
    lost_track_buffer           = 180,
    minimum_matching_threshold  = 0.1,
    minimum_consecutive_frames  = 3,
    asso_func                   = "diou",
    brownian_pos_noise          = 15,
    aspect_weight               = 0.0,
    behavioral_weight           = 0.0,
    vial_rois                   = _ablation_vials,
    det_log_csv                 = _DET_CACHE,
)

results = {}
with tempfile.TemporaryDirectory() as _tmp:
    for label, extra in [
        ("no_jump", dict(jump_factor=0.0, jump_iou_threshold=0.0, jump_inertia=0.0)),
        ("jump_x2", dict(jump_factor=2.0, jump_iou_threshold=0.05, jump_inertia=0.05)),
    ]:
        _csv = os.path.join(_tmp, label + ".csv")
        df, trk = export_tracks_xy_tuple_csv_one_config(
            output_csv=_csv, **_base_kwargs, **extra
        )
        long = wide_to_long(df)
        mean_cov = long.groupby("orig_id")["frame"].count().mean() / df.shape[0] * 100
        results[label] = {
            "raw_ids":        df.shape[1] - 1,
            "mean_dets":      sum(e[1] for e in trk.detection_log) / len(trk.detection_log),
            "mean_emitted":   sum(e[2] for e in trk.detection_log) / len(trk.detection_log),
            "mean_coverage%": round(mean_cov, 1),
            "suppressed":     len(trk.suppressed_tracks),
        }

print("%-22s %10s %10s" % ("Metric", "no_jump", "jump_x2"))
print("-" * 44)
for metric in ["raw_ids", "mean_dets", "mean_emitted", "mean_coverage%", "suppressed"]:
    a = results["no_jump"][metric]
    b = results["jump_x2"][metric]
    diff = "  (%+.1f)" % (b - a) if isinstance(b, float) else "  (%+d)" % (b - a)
    print("%-22s %10.1f %10.1f%s" % (metric, a, b, diff))
print("Expected flies: %d" % _N_EXPECTED)

## 9 — Multi-video jump ablation

Runs no_jump vs jump_x2 across three videos using their detection caches.
All other params held fixed at the run_68 baseline.
Prints a single summary table across all videos.

In [ ]:
import tempfile, os, json
import pandas as pd
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long

# ── Videos to test ───────────────────────────────────────────────────────
VIDEOS = [
    {
        "label":      "41DPE_n004",
        "pp_video":   "../outputs/run_68_41DPE_n004/2024-03-11_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_41d_004-converted_pp.mp4",
        "det_cache":  "../outputs/run_68_41DPE_n004/detections_raw.csv",
        "roi_json":   "../outputs/run_68_41DPE_n004/vial_rois.json",
        "n_expected": 42,
    },
    {
        "label":      "28DPE_n003",
        "pp_video":   "../outputs/run_77_28DPE_n003/2024-02-27_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_28d_003-converted_pp.mp4",
        "det_cache":  "../outputs/run_77_28DPE_n003/detections_raw.csv",
        "roi_json":   "../outputs/run_77_28DPE_n003/vial_rois.json",
        "n_expected": 42,
    },
    {
        "label":      "13DPE_n002",
        "pp_video":   "../outputs/run_80_13DPE_n002/2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_002-converted_pp.mp4",
        "det_cache":  "../outputs/run_80_13DPE_n002/detections_raw.csv",
        "roi_json":   "../outputs/run_80_13DPE_n002/vial_rois.json",
        "n_expected": 42,
    },
]

CONDITIONS = [
    ("no_jump", dict(jump_factor=0.0, jump_iou_threshold=0.05, jump_inertia=0.05)),
    ("jump_x2", dict(jump_factor=2.0, jump_iou_threshold=0.05, jump_inertia=0.05)),
]

rows = []
with tempfile.TemporaryDirectory() as _tmp:
    for vid in VIDEOS:
        with open(vid["roi_json"]) as f:
            vials = {k: tuple(v) for k, v in json.load(f).items()}
        base = dict(
            video_path                  = vid["pp_video"],
            api_key                     = API_KEY,
            model_id                    = MODEL_ID,
            detection_confidence_rfdetr = 0.4,
            confidence                  = 0.55,
            lost_track_buffer           = 180,
            minimum_matching_threshold  = 0.1,
            minimum_consecutive_frames  = 3,
            asso_func                   = "diou",
            brownian_pos_noise          = 15,
            aspect_weight               = 0.0,
            behavioral_weight           = 0.0,
            vial_rois                   = vials,
            det_log_csv                 = vid["det_cache"],
        )
        for cond, extra in CONDITIONS:
            _csv = os.path.join(_tmp, vid["label"] + "_" + cond + ".csv")
            df, trk = export_tracks_xy_tuple_csv_one_config(
                output_csv=_csv, **base, **extra
            )
            long = wide_to_long(df)
            mean_cov = long.groupby("orig_id")["frame"].count().mean() / df.shape[0] * 100
            rows.append({
                "video":      vid["label"],
                "condition":  cond,
                "n_expected": vid["n_expected"],
                "raw_ids":    df.shape[1] - 1,
                "id_error":   (df.shape[1] - 1) - vid["n_expected"],
                "coverage%":  round(mean_cov, 1),
                "suppressed": len(trk.suppressed_tracks),
            })
            print(f"{vid['label']}  {cond}  raw_ids={df.shape[1]-1}  coverage={mean_cov:.1f}%")

df_results = pd.DataFrame(rows)

# Pivot to wide format for easy comparison
df_pivot = df_results.pivot(index="video", columns="condition", values=["raw_ids", "id_error", "coverage%", "suppressed"])
df_pivot.columns = [f"{metric}_{cond}" for metric, cond in df_pivot.columns]
df_pivot["raw_ids_delta"]   = df_pivot["raw_ids_jump_x2"]   - df_pivot["raw_ids_no_jump"]
df_pivot["coverage%_delta"] = df_pivot["coverage%_jump_x2"] - df_pivot["coverage%_no_jump"]
df_pivot["n_expected"] = [v["n_expected"] for v in VIDEOS]

print()
print("%-14s %8s %8s %8s %9s %9s %10s" % ("video", "no_jump", "jump_x2", "delta", "cov_nojmp", "cov_jump", "cov_delta"))
print("%-14s %8s %8s %8s %9s %9s %10s" % ("", "ids", "ids", "ids", "%", "%", "%"))
print("-" * 70)
for vid in df_pivot.index:
    r = df_pivot.loc[vid]
    print("%-14s %8.0f %8.0f %+8.0f %9.1f %9.1f %+10.1f" % (
        vid,
        r["raw_ids_no_jump"], r["raw_ids_jump_x2"], r["raw_ids_delta"],
        r["coverage%_no_jump"], r["coverage%_jump_x2"], r["coverage%_delta"],
    ))
print()
print("Expected flies per video: 42")
